In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
!pip install -q kiwipiepy datasets accelerate evaluate sacrebleu bert_score

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import evaluate
from transformers import (
    BartForConditionalGeneration,
    PreTrainedTokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import re
from kiwipiepy import Kiwi
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast, TrainingArguments, Trainer
import torch
from transformers import BartTokenizer
import seaborn as sns
from datasets import load_dataset
from transformers import pipeline

### 데이터 로드

In [ ]:
dataset = load_dataset('Junhoee/Jeju-Standard-Translation')

In [ ]:
df = dataset['train'].to_pandas()
df.head()

### 데이터 필터링

In [ ]:
# 필터링 조건 설정
length_min = 5
length_max = 40
jaccard_min = 0.1
jaccard_max = 0.9

In [ ]:
# 모든 조건을 만족하는 행 필터링
filtered_df = dataset[
    (dataset['standard_len'] > length_min) & (dataset['standard_len'] <= length_max) &
    (dataset['dialect_len'] > length_min) & (dataset['dialect_len'] <= length_max) &
    (dataset['jaccard_similarity'] >= jaccard_min) & (dataset['jaccard_similarity'] <= jaccard_max)
]
filtered_df

In [ ]:
final_df = filtered_df[['dialect_form', 'standard_form']]
final_df

### 데이터 전처리

In [ ]:
kiwi = Kiwi()
tqdm.pandas()

# 전처리 함수 정의
def preprocess_text(text):
    # NaN 값 방지
    if pd.isna(text):
        return ""

    # Kiwi의 띄어쓰기 교정 기능 활용
    spacing_corrected_text = kiwi.space(text)

    # 한글, 숫자, 공백을 제외한 모든 문자 제거
    clean_text = re.sub(r'[^가-힣0-9\s]', '', spacing_corrected_text)

    # 여러 공백을 하나의 공백으로 치환하고 앞뒤 공백 제거
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

In [ ]:
# tqdm과 pandas.apply를 결합하여 전처리 진행
print("--- 전처리 시작 ---")
final_df['standard_form'] = final_df['standard_form'].progress_apply(preprocess_text)
final_df['dialect_form'] = final_df['dialect_form'].progress_apply(preprocess_text)
print("--- 전처리 완료 ---")

In [ ]:
final_df.to_csv('./제주도 프로젝트/final_preprocessed_data.csv', index=False, encoding='utf-8')

### 표준어 도메인 적응

In [ ]:
# dialect_form 컬럼을 input_text로, standard_form 컬럼을 target_text로 변경합니다.
final_df.rename(columns={'dialect_form': 'input_text', 'standard_form': 'target_text'}, inplace=True)

In [ ]:
df_standard_domain_adaptation = pd.DataFrame({
    'input_text': final_df['target_text'],
    'target_text': final_df['target_text']
})
df_standard_domain_adaptation

In [ ]:
# 데이터 분할 - 훈련 80%/검증 20%
train_df, valid_df = train_test_split(df_standard_domain_adaptation, test_size = 0.2, random_state = 42)

In [ ]:
# 분할된 데이터 크기 확인
print("--- 데이터 3분할 결과 ---")
print(f"전체 데이터프레임 크기: {df_standard_domain_adaptation.shape}")
print(f"훈련 데이터프레임 크기: {train_df.shape}")
print(f"검증 데이터프레임 크기: {valid_df.shape}")

--- 데이터 3분할 결과 ---
전체 데이터프레임 크기: (733122, 2)
훈련 데이터프레임 크기: (439872, 2)
검증 데이터프레임 크기: (146625, 2)
테스트 데이터프레임 크기: (146625, 2)


In [ ]:
# Pandas DataFrame을 Hugging Face Dataset 객체로 변환
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [ ]:
# 1. 모델 및 토크나이저 불러오기
model_name = 'gogamza/kobart-base-v2'
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
# 데이터 토크나이징 함수 정의
def tokenize_function(examples):
    model_inputs = tokenizer(examples['input_text'], max_length=60, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['target_text'], max_length=60, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])
tokenized_valid_dataset = valid_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
bertscore_metric = evaluate.load("bertscore")

In [ ]:
# 평가지표 함수 정의
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # 모델이 생성한 예측 토큰을 문자열로 디코딩
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # -100 패딩 값 처리 후 정답 토큰을 문자열로 디코딩
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ✅ 평가를 위해 예측과 정답 문장을 정규화합니다.
    decoded_preds = [re.sub(r'\s+', ' ', s).strip() for s in decoded_preds]
    decoded_labels = [re.sub(r'\s+', ' ', s).strip() for s in decoded_labels]

    # SacreBLEU 계산을 위해 참조 문장을 리스트의 리스트 형태로 변환
    decoded_labels_list = [[label] for label in decoded_labels]

    # SacreBLEU 점수 계산
    bleu_result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_list)

    # ✅ CHRF 점수 계산
    chrf_result = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_list)

    # ✅ BERTScore 점수 계산 (매우 느릴 수 있음)
    # lang='ko'를 지정하여 한국어에 최적화된 BERT 모델을 사용합니다.
    bertscore_result = bertscore_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang='ko'
    )

    # 모든 점수를 딕셔너리에 담아 반환합니다.
    result = {
        "sacrebleu": bleu_result["score"],
        "chrf": chrf_result["score"],
        "bertscore": np.mean(bertscore_result["f1"]) # F1 점수의 평균을 사용합니다.
    }

    return result

In [ ]:
# 학습 설정을 정의
training_args = Seq2SeqTrainingArguments(
    output_dir="./kobart_jeju_translation",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=512,
    per_device_eval_batch_size=512,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
    save_strategy="epoch",
    report_to='none',
    fp16=True, # ✅ 학습 시간 단축을 위한 혼합 정밀도 학습
    gradient_accumulation_steps=4, # ✅ 그래디언트 누적 단계 설정 (배치 사이즈를 2배로 사용하는 효과)
    predict_with_generate=True,
    generation_num_beams=1,
)

In [ ]:
# Seq2SeqTrainer 객체 정의
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
print("\n--- 표준어 -> 표준어 모델 학습 시작 (trainer) ---")
result = trainer.train()
print("\n--- 학습 완료 ---")

In [ ]:
# --- 학습된 모델 저장 ---
output_path = "./제주도 프로젝트/standard_model_1"
trainer.save_model(output_path)

### 제주어 도메인 적응

In [ ]:
df_dialect_domain_adaptation = pd.DataFrame({
    'input_text': final_df['input_text'],
    'target_text': final_df['input_text']
})
df_dialect_domain_adaptation

In [ ]:
train, valid_df = train_test_split(df_dialect_domain_adaptation, test_size = 0.2, random_state = 42)

In [ ]:
# 분할된 데이터 크기 확인
print("--- 데이터 3분할 결과 ---")
print(f"전체 데이터프레임 크기: {df_dialect_domain_adaptation.shape}")
print(f"훈련 데이터프레임 크기: {train_df.shape}")
print(f"검증 데이터프레임 크기: {valid_df.shape}")

--- 데이터 3분할 결과 ---
전체 데이터프레임 크기: (733122, 2)
훈련 데이터프레임 크기: (439872, 2)
검증 데이터프레임 크기: (146625, 2)
테스트 데이터프레임 크기: (146625, 2)


In [ ]:
# Pandas DataFrame을 Hugging Face Dataset 객체로 변환
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [ ]:
# 1. 모델 및 토크나이저 불러오기
model_path = './제주도 프로젝트/standard_model_1'
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


In [ ]:
# 2. 데이터 토크나이징 함수 정의
def tokenize_function(examples):
    model_inputs = tokenizer(examples['input_text'], max_length=60, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['target_text'], max_length=60, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])
tokenized_valid_dataset = valid_dataset.map(tokenize_function, batched=True, remove_columns=['input_text', 'target_text'])

Map:   0%|          | 0/439872 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/146625 [00:00<?, ? examples/s]

Map:   0%|          | 0/146625 [00:00<?, ? examples/s]

In [ ]:
# 학습 설정을 정의
training_args = Seq2SeqTrainingArguments(
    output_dir="./kobart_jeju_translation",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=512,
    per_device_eval_batch_size=512,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
    save_strategy="epoch",
    report_to='none',
    fp16=True, # ✅ 학습 시간 단축을 위한 혼합 정밀도 학습
    gradient_accumulation_steps=4, # ✅ 그래디언트 누적 단계 설정 (배치 사이즈를 2배로 사용하는 효과)
    predict_with_generate=True,
    generation_num_beams=4,
)

In [ ]:
# Seq2SeqTrainer 객체 정의
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
print("\n--- 제주어 -> 제주어 모델 학습 시작 (trainer) ---")
result = trainer.train()
print("\n--- 학습 완료 ---")

In [ ]:
# --- 학습된 모델 저장 ---
output_path = "./제주도 프로젝트/domain_model"
trainer.save_model(output_path)

### 도메인 적응 완료, 최종 번역 모델 파인튜닝 시작

In [ ]:
train_valid_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42)
train_df, valid_df = train_test_split(train_valid_df, test_size=0.25, random_state=42)

In [ ]:
# 분할된 데이터 크기 확인
print("--- 데이터 3분할 결과 ---")
print(f"전체 데이터프레임 크기: {final_df.shape}")
print(f"훈련 데이터프레임 크기: {train_df.shape}")
print(f"검증 데이터프레임 크기: {valid_df.shape}")
print(f"테스트 데이터프레임 크기: {test_df.shape}")

In [ ]:
# 4. 각 데이터셋에 대해 양방향 데이터 생성
def create_bidirectional_data(df):
    jeju_to_std = pd.DataFrame({
        'input_text': '[제주] ' + df['input_text'],
        'target_text': '[표준]' + df['target_text']
    })
    std_to_jeju = pd.DataFrame({
        'input_text': '[표준] ' + df['target_text'],
        'target_text': '[제주]' + df['input_text']
    })
    # 각 방향의 데이터를 합치고 무작위로 섞습니다.
    # 각 세트 내에서 섞기 때문에 데이터 누수 문제 없음.
    return pd.concat([jeju_to_std, std_to_jeju]).sample(frac=1).reset_index(drop=True)

In [ ]:
print("양방향 데이터 생성 및 결합 중...")
train_combined = create_bidirectional_data(train_df)
valid_combined = create_bidirectional_data(valid_df)
test_combined = create_bidirectional_data(test_df)
print("양방향 데이터 생성 및 결합 완료.")

In [ ]:
# ✅ 이 부분을 추가하여 데이터 크기를 확인합니다.
print("--- 최종 데이터셋 크기 확인 ---")
print(f"훈련 데이터셋 크기: {len(train_combined)}")
print(f"검증 데이터셋 크기: {len(valid_combined)}")
print(f"테스트 데이터셋 크기: {len(test_combined)}")

In [ ]:
# 분할된 데이터 크기 확인
print("--- 데이터 3분할 결과 ---")
print(f"훈련 데이터프레임 크기: {train_combined.shape}")
print(f"검증 데이터프레임 크기: {valid_combined.shape}")
print(f"테스트 데이터프레임 크기: {test_combined.shape}")

In [ ]:
# 결과 확인
print("\n최종 Train 데이터셋 예시:")
train_combined.head()

In [ ]:
print("\n최종 Test 데이터셋 예시:")
test_combined.head()

In [ ]:
# 1. 모델 및 토크나이저 불러오기
model_path = './제주도 프로젝트/domain_model'
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)

In [ ]:
# KoBART 토크나이저에 새로운 토큰 추가
new_tokens = ['[제주]', '[표준]']
num_added_tokens = tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))

print(f"추가된 토큰 개수: {num_added_tokens}")
print(f"새로운 토크나이저 어휘 크기: {len(tokenizer)}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


추가된 토큰 개수: 2
새로운 토크나이저 어휘 크기: 30002


In [ ]:
# 수정된 토크나이징 함수
def tokenize_function_with_tags(examples):
    # 토크나이저가 추가된 토큰을 인식하게 됩니다.
    model_inputs = tokenizer(examples['input_text'], max_length=60, truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['target_text'], max_length=60, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Pandas DataFrame을 Hugging Face Dataset 객체로 변환
train_dataset = Dataset.from_pandas(train_combined)
valid_dataset = Dataset.from_pandas(valid_combined)
test_dataset = Dataset.from_pandas(test_combined)

In [ ]:
tokenized_train_dataset = train_dataset.map(tokenize_function_with_tags, batched=True, remove_columns=['input_text', 'target_text'])
tokenized_valid_dataset = valid_dataset.map(tokenize_function_with_tags, batched=True, remove_columns=['input_text', 'target_text'])
tokenized_test_dataset = test_dataset.map(tokenize_function_with_tags, batched=True, remove_columns=['input_text', 'target_text'])

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
bertscore_metric = evaluate.load("bertscore")

In [ ]:
# 기존의 compute_metrics 함수 시작 부분

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 1. 제주어 -> 표준어 데이터와 표준어 -> 제주어 데이터를 분리합니다.
    jeju_to_std_preds = []
    jeju_to_std_labels = []
    std_to_jeju_preds = []
    std_to_jeju_labels = []

    for pred, label in zip(decoded_preds, decoded_labels):
        if label.startswith('[표준]'):
            jeju_to_std_preds.append(re.sub(r'\s+', ' ', pred).strip())
            jeju_to_std_labels.append(re.sub(r'\s+', ' ', label).strip())
        elif label.startswith('[제주]'):
            std_to_jeju_preds.append(re.sub(r'\s+', ' ', pred).strip())
            std_to_jeju_labels.append(re.sub(r'\s+', ' ', label).strip())

    # 2. SacreBLEU 계산을 위해 참조 문장을 리스트의 리스트 형태로 변환합니다.
    jeju_to_std_labels_list = [[label] for label in jeju_to_std_labels]
    std_to_jeju_labels_list = [[label] for label in std_to_jeju_labels]

    # 3. 각 방향별로 지표를 개별적으로 계산합니다.
    jeju_to_std_bleu = bleu_metric.compute(predictions=jeju_to_std_preds, references=jeju_to_std_labels_list)["score"]
    std_to_jeju_bleu = bleu_metric.compute(predictions=std_to_jeju_preds, references=std_to_jeju_labels_list)["score"]

    jeju_to_std_chrf = chrf_metric.compute(predictions=jeju_to_std_preds, references=jeju_to_std_labels_list)["score"]
    std_to_jeju_chrf = chrf_metric.compute(predictions=std_to_jeju_preds, references=std_to_jeju_labels_list)["score"]

    jeju_to_std_bertscore = np.mean(bertscore_metric.compute(predictions=jeju_to_std_preds, references=jeju_to_std_labels, lang='ko')["f1"])
    std_to_jeju_bertscore = np.mean(bertscore_metric.compute(predictions=std_to_jeju_preds, references=std_to_jeju_labels, lang='ko')["f1"])

    # 4. 각 방향별로 분리된 결과를 반환합니다.
    result = {
        "jeju_to_standard_sacrebleu": jeju_to_std_bleu,
        "standard_to_jeju_sacrebleu": std_to_jeju_bleu,
        "jeju_to_standard_chrf": jeju_to_std_chrf,
        "standard_to_jeju_chrf": std_to_jeju_chrf,
        "jeju_to_standard_bertscore_f1": jeju_to_std_bertscore,
        "standard_to_jeju_bertscore_f1": std_to_jeju_bertscore
    }

    return result

In [ ]:
# 학습 설정을 정의
training_args = Seq2SeqTrainingArguments(
    output_dir="./kobart_jeju_translation",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
    save_strategy="epoch",
    report_to='none',
    fp16=True, # 학습 시간 단축을 위한 혼합 정밀도 학습
    gradient_accumulation_steps=16, # 그래디언트 누적 단계 설정 (배치 사이즈를 2배로 사용하는 효과)
    predict_with_generate=True,
    generation_num_beams=5,
    generation_max_length=128,
)

In [ ]:
# Seq2SeqTrainer 객체 정의
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
print("\n--- 모델 학습 시작 (trainer) ---")
result = trainer.train()
print("\n--- 학습 완료 ---")

In [ ]:
# --- 학습된 모델 저장 ---
output_path = "./제주도 프로젝트/real_final_model3"
trainer.save_model(output_path)

### 추론해보자

In [ ]:
# 1. 학습된 모델과 토크나이저 로드
model_path = "./제주도 프로젝트/real_final_model3"
model = BartForConditionalGeneration.from_pretrained(model_path)
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_path)

In [ ]:
# 2. 파이프라인 생성
# task를 'translation'으로 설정하고, 학습된 모델과 토크나이저를 전달합니다.
translator = pipeline(
    'translation',
    model=model,
    tokenizer=tokenizer
)

In [ ]:
# 3. 추론할 문장 준비 (태그 포함)
# 제주어 -> 표준어
jeju_sentence = '[제주] 니 혼자만 속상해 허지 말앙, 나한테도 고라봅써.'
# 표준어 -> 제주어
standard_sentence = '[표준] 너 혼자만 속상해하지 말고, 나한테도 말해봐.'

In [ ]:
# 4. 추론 실행
result_jeju_to_standard = translator(jeju_sentence, max_length=128)
result_standard_to_jeju = translator(standard_sentence, max_length=128)

print("--- 제주어 -> 표준어 추론 결과 ---")
print(f"입력: {jeju_sentence}")
print(f"출력: {result_jeju_to_standard[0]['translation_text']}")

print("\n--- 표준어 -> 제주어 추론 결과 ---")
print(f"입력: {standard_sentence}")
print(f"출력: {result_standard_to_jeju[0]['translation_text']}")

### test_dataset으로 평가

In [ ]:
# test dataset으로 평가
predict_args = Seq2SeqTrainingArguments(
    output_dir="./prediction_output",
    per_device_eval_batch_size=512, # 예측 배치 사이즈 설정
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=128,
)

In [ ]:
# Trainer 객체 정의 및 예측 실행
trainer = Seq2SeqTrainer(
    model=model,
    args=predict_args,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# predict 메서드로 테스트 데이터셋 평가
results = trainer.predict(tokenized_test_dataset)

In [ ]:
print("\n--- 최종 예측 및 평가 완료 ---")
print(results.metrics)

In [ ]:
#  예측 문장, 정답 문장, 입력 문장을 함께 출력 (추가할 코드)
print("\n--- 샘플 예측 결과 ---")
preds = results.predictions[0] if isinstance(results.predictions, tuple) else results.predictions
labels = results.label_ids

In [ ]:
# 토큰을 문장으로 디코딩
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

In [ ]:
# 원본 입력 문장 가져오기 (pandas dataframe에서 직접)
original_inputs = test_df['input_text'].tolist()

In [ ]:
# 예측 결과 샘플 10개 출력 (전체 데이터가 많으므로 일부만 확인)
num_samples_to_show = 10
for i in range(num_samples_to_show):
    # 입력 문장에는 제주/표준 태그가 포함되어 있으므로 그대로 출력
    print(f"입력: {original_inputs[i]}")
    # 예측 및 정답 문장 출력
    print(f"예측: {decoded_preds[i]}")
    print(f"정답: {decoded_labels[i]}")
    print("-" * 50)

print("✅ 모든 과정 완료!")